# Bankruptcy Prediction


# Problem description

You are to predict whether a company will go bankrupt in the following
year, based on financial attributes of the company.

Perhaps you are contemplating lending money to a company, and need to know
whether the company is in near-term danger of not being able to repay.


## Learning objectives

- Demonstrate mastery on solving a classification problem and presenting
your end-to-end approach in a notebook.
- We will make suggestions for ways to approach the problem
    - But there will be little explicit direction for this task.
- It is meant to be analogous to a pre-interview task that a potential
employer might assign to verify your skill.


# Before we start: why do companies go bankrupt?

At the simplest level, a company runs into trouble when the money coming
in isn't enough to cover the money it must pay out.

- **Money coming in**: revenue from sales, and the profit left over after
costs.
- **Money going out**: the costs of doing business, and -- critically --
debts coming due, often with interest attached.
- **The cushion in between**: cash and other liquid resources on hand, and
the equity buffer built up by owners over time.

A company heading toward bankruptcy typically shows some mix of: shrinking
or negative profit, growing obligations, and a thinning cushion. The
dataset gives you dozens of financial attributes that each touch on one
piece of this picture. Part of your job is figuring out which pieces
matter most, and how they combine.


# A light glossary of financial terms

You don't need an accounting background to do well on this assignment --
recognizing patterns in the data matters more than knowing every term by
heart. But since the attributes use real financial-statement language,
here's a plain-English reference.

## Revenue & Income (money the company earns)
- **sales** -- total revenue generated from selling products/services
during the year.
- **Gross Profit** -- what's left from sales after subtracting the direct
cost of producing what was sold.
- **Net Profit** -- the bottom line: what's left after *all* expenses,
interest, and taxes are paid.
- **Operating Profit** -- profit from normal day-to-day business, before
financing costs or one-off items.
- **Profit on Sales** -- profit tied specifically to sales activity.
- **EBIT** -- profit before interest and taxes are subtracted, a way to
look at the core business regardless of how it's financed or taxed.
- **EBITDA** -- like EBIT, but also before non-cash accounting charges like
depreciation -- often used as a rough stand-in for cash the business
generates.
- A few other columns are close *flavors of* gross profit -- for example,
with interest or depreciation added back in, or totaled over three years
instead of one. Treat these as closely related variations; the specific
adjustment matters less than recognizing they're all measuring
profitability.

## Costs & Obligations (money the company owes or spends)
- **total_liabilities** -- everything the company owes, both short- and
long-term.
- **short_term_liabilities** -- obligations due within the coming year
(some formulas below call this "current liabilities" -- same thing).
- **Operating Expenses** -- the everyday costs of running the business.
- **Cost of Products Sold** -- the direct cost of producing or acquiring
the goods a company actually sold (sometimes abbreviated COGS).
- **Financial Expenses** -- the cost of financing, mainly interest paid
on borrowed money.
- **Liabilities Net of Cash** -- what the company owes, minus the cash it
has on hand to help pay it off.

## Ownership & Capital (money invested by owners, not borrowed)
- **Owner's Equity** -- the value of the company belonging to its owners
after all debts are subtracted.
- **Retained Earnings** -- accumulated profit kept in the business over
time rather than paid out to owners.
- A couple of other columns are close *flavors of* equity -- one reflects
equity built up from operations rather than what owners first put in,
another (sometimes called "constant capital") combines equity with
long-term borrowing into a single "funds we don't need to repay soon"
figure. These are related capital measures, not fundamentally different
ideas.

## Resources & Assets (what the company has on hand or owns longer-term)
- **total_assets** -- the total value of everything the company owns.
- **Current Assets** -- assets expected to be converted to cash or used up
within a year (things like inventory, receivables, and cash itself).
- **Fixed Assets** -- longer-term physical resources, like property and
equipment, not expected to convert to cash within a year.
- **Inventory** -- the value of goods held for sale.
- **Accounts Receivable** -- money customers owe the company for goods or
services already delivered.
- **Working Capital** -- the cushion left over after covering near-term
bills with short-term assets.
- A couple of other columns are stricter *flavors of* "current assets" --
some assets convert to cash faster than others, so these exclude inventory
(and sometimes receivables too) to focus on the most liquid resources.

A handful of attributes are given to you as ready-made ratios (their
exact formula is listed further below, near where the data is loaded).
Not every provided attribute is necessarily trustworthy -- where you can
cross-check a value against other information you have, it's worth doing.


# Import modules


In [ ]:
## Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn

import os
import math

%matplotlib inline


# Get the data

- Each example is a row of data corresponding to a single company
- The column `Bankrupt` is 1 if the company subsequently went bankrupt; 0
if it did not go bankrupt
- The column `Id` is a Company Identifier
- The other columns (`num_01`, `num_02`, ..., `ratio_01`, `ratio_02`, ...,
`total_assets`, `sales`, `total_liabilities`, `short_term_liabilities`)
are financial attributes for the company; see the glossary above and the
attribute table further below
- `Id` is just a company identifier, not a feature -- don't feed it to
your model


In [ ]:
# Data directory
DATA_DIR = "./Data"

data_file = "companies.csv"
data = pd.read_csv( os.path.join(DATA_DIR, "train", data_file) )

target_attr = "Bankrupt"
feature_cols = [c for c in data.columns if c not in (target_attr, "Id")]

n_samples, n_attrs = data.shape
print("Data shape: ", data.shape)


## Have a look at the data

Here's a peek.


In [ ]:
data.head()


In [ ]:
data.info()


## Attributes given as raw figures

Each `num_##` column is a single financial figure for the company. Its
name below tells you which -- look it up in the glossary above for what
it means:

| Attribute | Financial Quantity |
|---|---|
| num_01 | Gross Profit Plus Depreciation |
| num_02 | EBITDA |
| num_03 | Gross Profit |
| num_04 | Operating Profit |
| num_05 | Gross Profit |
| num_06 | Current Assets |
| num_07 | Gross Profit Plus Interest |
| num_08 | Long-Term Capital Base |
| num_09 | Operating Profit |
| num_10 | Inventory |
| num_11 | Gross Profit (3-Year Cumulative) |
| num_12 | Near-Cash Assets |
| num_13 | Net Profit |
| num_14 | Owner's Equity |
| num_15 | Net Profit |
| num_16 | Net Profit Plus Depreciation |
| num_17 | Gross Profit (Adjusted) |
| num_18 | Profit on Sales |
| num_19 | Working Capital |
| num_20 | Retained Earnings |
| num_21 | Operating Expenses |
| num_22 | EBIT |
| num_23 | Current Assets |
| num_24 | EBITDA |
| num_25 | Gross Profit |
| num_26 | Owner's Equity (Beyond Initial Capital) |
| num_27 | Accounts Receivable |
| num_28 | Liabilities Net of Cash |
| num_29 | Owner's Equity |
| num_30 | Gross Profit Plus Interest |
| num_31 | Profit on Sales |
| num_32 | Quick Assets |
| num_33 | Gross Profit |
| num_34 | Gross Profit Plus Depreciation |
| num_35 | Operating Expenses |


## Attributes given as ratios

Some of the attributes are given to you as ready-made ratios. Their
formulas:

| Attribute | Formula |
|---|---|
| ratio_01 | (current assets - inventories) / long-term liabilities |
| ratio_02 | logarithm of total assets |
| ratio_03 | [(cash + short-term securities + receivables - short-term liabilities) / (operating expenses - depreciation)] * 365 |
| ratio_04 | long-term liabilities / equity |
| ratio_05 | profit on operating activities / financial expenses |
| ratio_06 | equity / fixed assets |
| ratio_07 | sales / fixed assets |
| ratio_08 | rotation receivables + inventory turnover in days |
| ratio_09 | total costs / total sales |
| ratio_10 | net profit / inventory |
| ratio_11 | total sales / total assets |
| ratio_12 | (short-term liabilities * 365) / cost of products sold |
| ratio_13 | (short-term liabilities * 365) / sales |
| ratio_14 | constant capital / fixed assets |
| ratio_15 | total assets / total liabilities |
| ratio_16 | sales / total assets |
| ratio_17 | sales / receivables |
| ratio_18 | (current liabilities * 365) / cost of products sold |
| ratio_19 | working capital / fixed assets |
| ratio_20 | sales / short-term liabilities |
| ratio_21 | sales (n) / sales (n-1) |
| ratio_22 | (total liabilities * 365) / (gross profit + depreciation) |
| ratio_23 | (current assets - inventory - short-term liabilities) / (sales - gross profit - depreciation) |
| ratio_24 | sales / inventory |
| ratio_25 | total liabilities / ((profit on operating activities + depreciation) * (12/365)) |
| ratio_26 | working capital |
| ratio_27 | (inventory * 365) / cost of products sold |


# Evaluating your project

We will evaluate your submission on a test dataset that we provide
- It has no labels, so **you** can't use it to evaluate your model, but
**we** have the labels
- We will call this evaluation dataset the "holdout" data

Let's get it.


In [ ]:
holdout_data = pd.read_csv( os.path.join(DATA_DIR, "holdout", data_file) )

print("Data shape: ", holdout_data.shape)


We will evaluate your model on the holdout examples using metrics
- Accuracy
- Recall
- Precision

You will likely have to make a trade-off between Recall and Precision.
Our evaluation of your submission will be partially based on how you made
(and described) that trade-off.

You may assume that missing an actual bankruptcy (a false negative) is 5
times worse than a false alarm (a false positive) -- i.e., failing to
flag a company that goes bankrupt costs 5x what incorrectly flagging a
healthy company costs.


# Establish a baseline

Before building your real model, establish a simple baseline: one that
ignores the given features entirely and always predicts the majority
class. This gives you a concrete lower bound -- any real model should
beat it by a meaningful margin, and comparing against it as you go (not
just at the end) is a standard, useful sanity check.

The train/test split and `cost` function below are reused for every
evaluation from here on, including your final model in the Submission
Summary -- so keep using these same `X_train`/`X_test`/`y_train`/`y_test`
(rather than creating a new split later) to keep every comparison you
make apples-to-apples.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier

X_train, X_test, y_train, y_test = train_test_split(
    data.drop(columns=[target_attr]), data[target_attr], test_size=0.2,
    random_state=42, stratify=data[target_attr])

def cost(y_true, y_pred, fn_weight=5, fp_weight=1):
    """Default cost function matching the stated trade-off (5x worse to
    miss a bankruptcy). Adapt this if you use a different formulation --
    just apply whichever version you use consistently to both the
    baseline and your final model, so the ratio you report later stays
    meaningful."""
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return fn_weight * fn + fp_weight * fp

baseline_pipeline = DummyClassifier(strategy="most_frequent")
baseline_pipeline.fit(X_train, y_train)  # DummyClassifier ignores X entirely (incl. Id) --
                                          # your own final_pipeline below does need to ignore Id itself

baseline_cost = cost(y_test, baseline_pipeline.predict(X_test))
print(f"Baseline cost (always predicts the majority class): {baseline_cost}")


# Your model

Time for you to continue on your own.


## Submission guidelines

Although your notebook may contain many models (e.g., due to your
iterative development) we will only evaluate a single model.

So choose one and obey the following guidelines. Add a markdown cell
right before `final_pipeline` is built explaining which model you chose
and why.


### What you submit: `final_pipeline`

Rather than asking you to hand-write a standalone prediction function, we
ask that your notebook define a single fitted object:

    final_pipeline

`final_pipeline` must support `.predict(X)`, where `X` is a raw pandas
DataFrame in exactly the same format as the data you were given -- same
columns (including `Id`), same dtypes -- including the holdout data
(`holdout_data`, loaded earlier -- we call it `X_hold` below to mirror
`X_train`/`X_test` naming, but it's the same object), which has no
`Bankrupt` column. If your approach can also produce probability
estimates (`.predict_proba(X)`), that's useful but not required -- any
classifier is acceptable, including ones that only produce hard class
predictions.

**We strongly recommend building `final_pipeline` as an
`sklearn.pipeline.Pipeline`** (optionally combined with
`ColumnTransformer`), with every step you need bundled in: any features
you engineer from the given columns, imputation, scaling, and finally your
classifier of choice. The final step doesn't have to be a standard
sklearn classifier -- a custom class is fine, as long as the object
`final_pipeline` as a whole exposes `.predict(X)` (and, optionally,
`.predict_proba(X)`).

The reason to do it this way, rather than writing your own function, is
that a fitted `Pipeline` *guarantees* the same transformations you apply
to your training data get applied identically to the holdout data when we
call `.predict()` -- there's no way for your train-time and
inference-time processing to accidentally drift apart, which is the main
risk in an exercise like this.

If part of your feature engineering doesn't fit neatly into a standard
sklearn transformer, you can wrap it in a
`sklearn.preprocessing.FunctionTransformer`, or write a small custom
transformer class with `fit`/`transform` methods -- both plug directly
into a `Pipeline`.

**Fit `final_pipeline` on `X_train`/`y_train`** (from the baseline
section above), not on all of `data` -- this keeps the Train/Test
comparison in the Submission Summary meaningful, since `X_test` stays
genuinely unseen by the exact object being graded.

We will evaluate your submission by calling

    y_hold_pred = final_pipeline.predict(X_hold)

to compute Accuracy, Recall, and Precision against the true holdout
labels, which only the instructors have. Since `.predict()` is what
determines your reported metrics, make sure its output reflects the
Recall/Precision trade-off you've decided on -- if `final_pipeline` also
exposes `.predict_proba(X)`, that trade-off might be baked in as a chosen
threshold; if not, it needs to be reflected however your method decides
between classes.

Before considering your notebook done, actually run
`final_pipeline.predict(holdout_data)` yourself (further down, in the
Submission Summary) and confirm it returns a prediction for every row
without error -- catching a shape/dtype mismatch yourself is much better
than discovering one at grading time.


### Notebook style

Your notebook will probably have lots of code that does not contribute to
building `final_pipeline`.

We don't want to have to run the entire notebook or figure out which cells
are prerequisites. As a matter of good coding practice, please organize
your notebook (and use markdown) so that a single, clearly-marked run
through the cells builds `final_pipeline` from scratch, without requiring
us to untangle your exploratory work. You don't need to delete your
exploratory cells -- labeling them clearly (e.g. under an "## Exploration"
heading, separate from the cells that actually build `final_pipeline`) is
enough.


# Submission summary (required)

This section is **required**, not optional -- we grade directly from what
it prints, so make sure it's reproducible by running the notebook top to
bottom. It has several parts: a performance table (train vs. your
held-out test split, reusing `X_train`/`X_test`/`y_train`/`y_test` from
the baseline section above), a baseline comparison, a feature-correlation
check, a final sanity check against the real holdout file, and a short
written report. Fill in every blank.


In [ ]:
# Uses X_train/X_test/y_train/y_test from the "Establish a baseline"
# section above. X_train/X_test include `Id` -- the same format
# final_pipeline.predict() will actually be called with -- your pipeline
# should ignore Id internally, the same way it will need to for the real
# holdout call.
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                              roc_auc_score, average_precision_score)

y_train_pred = final_pipeline.predict(X_train)
y_test_pred = final_pipeline.predict(X_test)

print(f"{'Split':8s} {'Accuracy':>10s} {'Recall':>10s} {'Precision':>10s}")
for split_name, y_true, y_pred in [("Train", y_train, y_train_pred), ("Test", y_test, y_test_pred)]:
    print(f"{split_name:8s} {accuracy_score(y_true, y_pred):>10.1%} "
          f"{recall_score(y_true, y_pred):>10.1%} {precision_score(y_true, y_pred):>10.1%}")

# ROC-AUC / PR-AUC (train vs test) if your final_pipeline supports
# predict_proba -- these are the clearest signal of overfitting (compare
# the train row to the test row) and of ranking quality under class
# imbalance. If your model doesn't support predict_proba, this is skipped
# automatically -- explain why in the report below instead.
if hasattr(final_pipeline, "predict_proba"):
    proba_train = final_pipeline.predict_proba(X_train)[:, 1]
    proba_test = final_pipeline.predict_proba(X_test)[:, 1]
    print(f"\n{'Split':8s} {'ROC-AUC':>10s} {'PR-AUC':>10s}")
    print(f"{'Train':8s} {roc_auc_score(y_train, proba_train):>10.4f} "
          f"{average_precision_score(y_train, proba_train):>10.4f}")
    print(f"{'Test':8s} {roc_auc_score(y_test, proba_test):>10.4f} "
          f"{average_precision_score(y_test, proba_test):>10.4f}")
else:
    print("\nfinal_pipeline has no predict_proba -- ROC-AUC/PR-AUC not reported "
          "(explain why in the report below).")


In [ ]:
# Baseline comparison, using the same `cost` function and test split as
# the baseline established earlier.

final_cost = cost(y_test, y_test_pred)
improvement_ratio = baseline_cost / final_cost if final_cost > 0 else float("inf")

print(f"Baseline cost:        {baseline_cost}")
print(f"Final model cost:     {final_cost}")
print(f"Improvement ratio (baseline / final): {improvement_ratio:.2f}")


In [ ]:
# Highest pairwise correlation among your final feature set. Set
# X_features to whatever DataFrame of columns you actually feed your
# model (post feature-engineering, pre-imputation/scaling is fine).

X_features = X_train[feature_cols]  # YOUR CODE HERE: replace if your features differ from the raw columns

corr = X_features.corr(numeric_only=True).abs().to_numpy(copy=True)
np.fill_diagonal(corr, 0)
corr = pd.DataFrame(corr, index=X_features.columns, columns=X_features.columns)
max_corr = corr.to_numpy().max()
worst_pair = corr.stack().idxmax()
print(f"Highest pairwise correlation among final features: {max_corr:.4f} ({worst_pair[0]} vs {worst_pair[1]})")


In [ ]:
# Final sanity check: run final_pipeline against the actual holdout file,
# exactly as it will be called for grading. This should complete with no
# error and return one prediction per row -- if it doesn't, fix that
# before considering the notebook done.

X_hold = holdout_data
y_hold_pred = final_pipeline.predict(X_hold)
assert len(y_hold_pred) == len(X_hold), "predict() didn't return one row per holdout example"
print(f"OK -- predicted {len(y_hold_pred)} rows, bankrupt rate {y_hold_pred.mean():.1%}")


In [ ]:
# Written report -- fill in every field. Each is read directly for
# grading, so answer specifically enough that someone re-reading your
# notebook (without re-running your search/tuning process) can verify it.

submission_summary = {
    "approach_summary": "",
    # Briefly summarize your modeling process: what did you try, in what
    # order, and what did you learn or change at each step? (e.g. which
    # models/approaches you compared, what looking at your errors showed
    # you, and how that shaped what you tried next.) A few sentences or a
    # short list is fine -- however extensive your actual process was.

    "model_type": "",
    # e.g. "RandomForestClassifier" or "LogisticRegression"

    "model_size": "",
    # State the complexity of your FINAL model concretely: for a tree
    # ensemble, number of trees and max_depth; for a linear model,
    # penalty type, regularization strength, and number of nonzero
    # coefficients; for PCA or another dimensionality-reduction step (if
    # used), number of components retained. Use whatever is the
    # analogous complexity measure for your model type. Comment briefly
    # on how this compares to your train-vs-test numbers above.

    "feature_engineering_summary": "",
    # 2-4 sentences: what transformations or new features (if any) did
    # you create from the given attributes, and why (or why not)?

    "imbalance_handling": "",
    # How, if at all, did you address the class imbalance in the
    # training data?

    "cost_ratio_handling": "",
    # Explain specifically how final_pipeline.predict()'s output
    # reflects the stated cost asymmetry between false negatives and
    # false positives (e.g., a chosen decision threshold and how you
    # picked it, or another mechanism).
}

for key, value in submission_summary.items():
    print(f"{key}:\n  {value}\n")
